# Phase 1 and 2 - Data Acquisition and Preparation

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [2]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ROOT,
    RAW,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    UA
)

print("Project root :", ROOT)
print("Raw data     :", RAW)
print("CFPB         :", CFPB_RAW)
print("eCFR         :", ECFR_RAW)
print("PDF          :", PDF_RAW)
print("Processed    :", PROC)

assert CFPB_RAW.exists()
assert ECFR_RAW.exists()
assert PDF_RAW.exists()

print("\nPASS: Phase 1 directories are ready.")

Project root : /home/sagemaker-user/complaintiq
Raw data     : /home/sagemaker-user/complaintiq/data/raw
CFPB         : /home/sagemaker-user/complaintiq/data/raw/cfpb
eCFR         : /home/sagemaker-user/complaintiq/data/raw/ecfr
PDF          : /home/sagemaker-user/complaintiq/data/raw/pdf
Processed    : /home/sagemaker-user/complaintiq/data/processed

PASS: Phase 1 directories are ready.


In [3]:
download_script = PROJECT_ROOT / "src" / "download_data.py"

print("File:", download_script)
print("=" * 80)

print(download_script.read_text(encoding="utf-8"))

File: /home/sagemaker-user/complaintiq/src/download_data.py



In [4]:
from pathlib import Path

script = r'''from __future__ import annotations

import csv
import io
import re
import zipfile
from datetime import datetime
from html.parser import HTMLParser
from urllib.parse import urljoin

import requests

from src.config import ROOT, CFPB_RAW, UA

ARCHIVE_DIR = ROOT / "data" / "raw" / "cfpb_archive"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_PAGE = (
    "https://www.consumerfinance.gov/foia-requests/"
    "foia-electronic-reading-room/"
    "cfpb-consumer-complaint-database-narratives-archive/"
)

EXPORTS = {
    "CCDB Export January 2026 through February 2026":
        "january_february_2026.zip",
    "CCDB Export March 2026":
        "march_2026.zip",
    "CCDB Export April 2026":
        "april_2026.zip",
    "CCDB Export May 2026":
        "may_2026.zip",
    "CCDB Export June 2026":
        "june_2026.zip",
}

START_DATE = datetime(2026, 1, 1).date()
END_DATE = datetime(2026, 6, 30).date()

PRODUCT_OUTPUTS = {
    "Credit card": CFPB_RAW / "credit_card.csv",
    "Checking or savings account":
        CFPB_RAW / "checking_or_savings_account.csv",
}

REQUIRED_COLUMNS = {
    "Date received",
    "Product",
    "Consumer complaint narrative",
}


class ArchiveLinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self.current_href = None
        self.current_text = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            self.current_href = dict(attrs).get("href")
            self.current_text = []

    def handle_data(self, data):
        if self.current_href is not None:
            self.current_text.append(data)

    def handle_endtag(self, tag):
        if tag.lower() == "a" and self.current_href is not None:
            text = " ".join(" ".join(self.current_text).split())
            self.links.append((text, self.current_href))
            self.current_href = None
            self.current_text = []


def parse_received_date(value):
    value = (value or "").strip()
    if not value:
        return None

    for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%m/%d/%y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            pass

    return None


def download_archives():
    headers = dict(UA)
    headers.setdefault("User-Agent", "ComplaintIQ-capstone/1.0")

    print("Reading official CFPB narrative archive:")
    print(ARCHIVE_PAGE)

    response = requests.get(
        ARCHIVE_PAGE,
        headers=headers,
        timeout=(20, 120),
    )
    response.raise_for_status()

    parser = ArchiveLinkParser()
    parser.feed(response.text)

    available_links = {
        " ".join(text.split()).casefold(): urljoin(ARCHIVE_PAGE, href)
        for text, href in parser.links
        if href
    }

    for label, filename in EXPORTS.items():
        url = available_links.get(label.casefold())

        if not url:
            raise RuntimeError(
                f"Could not find official archive link: {label}. "
                "Open the CFPB archive page and verify the export label."
            )

        destination = ARCHIVE_DIR / filename

        # Reuse a previously downloaded archive only if it is a valid ZIP.
        if destination.exists() and zipfile.is_zipfile(destination):
            print(f"Archive already present and valid: {destination.name}")
            continue

        print(f"Downloading: {label}")

        download = requests.get(
            url,
            headers=headers,
            timeout=(30, 300),
        )
        download.raise_for_status()

        if not zipfile.is_zipfile(io.BytesIO(download.content)):
            raise RuntimeError(
                f"The CFPB download for '{label}' was not a valid ZIP. "
                f"HTTP status: {download.status_code}; "
                f"content type: {download.headers.get('Content-Type')}"
            )

        temporary = destination.with_suffix(".zip.part")
        temporary.write_bytes(download.content)
        temporary.replace(destination)

        print(
            f"Saved {destination.name}: "
            f"{destination.stat().st_size:,} bytes"
        )


def process_archives():
    for output in PRODUCT_OUTPUTS.values():
        output.parent.mkdir(parents=True, exist_ok=True)
        if output.exists():
            output.unlink()

    counts = {product: 0 for product in PRODUCT_OUTPUTS}
    monthly_counts = {}

    for label, filename in EXPORTS.items():
        archive_path = ARCHIVE_DIR / filename

        if not archive_path.exists() or not zipfile.is_zipfile(archive_path):
            raise RuntimeError(f"Missing or invalid archive: {archive_path}")

        with zipfile.ZipFile(archive_path) as archive:
            csv_names = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv")
            ]

            if len(csv_names) != 1:
                raise RuntimeError(
                    f"Expected exactly one CSV in {filename}; "
                    f"found: {csv_names}"
                )

            csv_name = csv_names[0]
            print(f"\nProcessing {filename} -> {csv_name}")

            with archive.open(csv_name) as binary_file:
                text_file = io.TextIOWrapper(
                    binary_file,
                    encoding="utf-8-sig",
                    newline="",
                )
                reader = csv.DictReader(text_file)

                if not reader.fieldnames:
                    raise RuntimeError(f"No CSV header in {filename}")

                missing = REQUIRED_COLUMNS - set(reader.fieldnames)
                if missing:
                    raise RuntimeError(
                        f"{filename} is missing required columns: {missing}. "
                        f"Found: {reader.fieldnames}"
                    )

                # Open outputs in append mode; write the header only once.
                with (
                    PRODUCT_OUTPUTS["Credit card"].open(
                        "a", encoding="utf-8-sig", newline=""
                    ) as credit_file,
                    PRODUCT_OUTPUTS["Checking or savings account"].open(
                        "a", encoding="utf-8-sig", newline=""
                    ) as checking_file,
                ):
                    writers = {
                        "Credit card": csv.DictWriter(
                            credit_file, fieldnames=reader.fieldnames
                        ),
                        "Checking or savings account": csv.DictWriter(
                            checking_file, fieldnames=reader.fieldnames
                        ),
                    }

                    for product, writer in writers.items():
                        if counts[product] == 0:
                            writer.writeheader()

                    for row in reader:
                        received = parse_received_date(row.get("Date received"))
                        product = (row.get("Product") or "").strip()
                        narrative = (
                            row.get("Consumer complaint narrative") or ""
                        ).strip()

                        if received is None or not (
                            START_DATE <= received <= END_DATE
                        ):
                            continue

                        if product not in PRODUCT_OUTPUTS:
                            continue

                        if not narrative:
                            continue

                        writers[product].writerow(row)
                        counts[product] += 1
                        month_key = received.strftime("%Y-%m")
                        key = (filename, month_key, product)
                        monthly_counts[key] = monthly_counts.get(key, 0) + 1

    print("\n=== Output summary ===")
    for product, output in PRODUCT_OUTPUTS.items():
        print(f"{product}: {counts[product]:,} rows")
        print(f"  File: {output}")
        if not output.exists() or output.stat().st_size == 0:
            raise RuntimeError(f"Output file is empty or missing: {output}")

    print("\n=== Monthly counts by source archive ===")
    for (archive, month, product), count in sorted(monthly_counts.items()):
        print(f"{archive} | {month} | {product}: {count:,}")

    print("\nPASS: CFPB narrative archive processing completed.")


def main():
    download_archives()
    process_archives()


if __name__ == "__main__":
    main()
'''

script_path = PROJECT_ROOT / "src" / "download_data.py"
script_path.write_text(script, encoding="utf-8")
print("Wrote:", script_path)
print("Bytes:", script_path.stat().st_size)

Wrote: /home/sagemaker-user/complaintiq/src/download_data.py
Bytes: 8497


In [5]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "src.download_data"],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    timeout=1800,
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"CFPB acquisition failed with exit code {result.returncode}. "
        "Review the output above before retrying."
    )

Reading official CFPB narrative archive:
https://www.consumerfinance.gov/foia-requests/foia-electronic-reading-room/cfpb-consumer-complaint-database-narratives-archive/
Downloading: CCDB Export January 2026 through February 2026
Saved january_february_2026.zip: 36,832,036 bytes
Downloading: CCDB Export March 2026
Saved march_2026.zip: 22,053,643 bytes
Downloading: CCDB Export April 2026
Saved april_2026.zip: 22,963,936 bytes
Downloading: CCDB Export May 2026
Saved may_2026.zip: 21,033,928 bytes
Downloading: CCDB Export June 2026
Saved june_2026.zip: 19,624,767 bytes

Processing january_february_2026.zip -> CCDB_Export_15_January_2026_through_February_2026.csv

Processing march_2026.zip -> CCDB_Export_16_March_2026.csv

Processing april_2026.zip -> CCDB_Export_17_April_2026.csv

Processing may_2026.zip -> CCDB_Export_18_May_2026.csv

Processing june_2026.zip -> CCDB_Export_19_June_2026.csv

=== Output summary ===
Credit card: 18,613 rows
  File: /home/sagemaker-user/complaintiq/data/raw

In [6]:
import pandas as pd

outputs = {
    "Credit card": CFPB_RAW / "credit_card.csv",
    "Checking or savings account":
        CFPB_RAW / "checking_or_savings_account.csv",
}

for product, path in outputs.items():
    df = pd.read_csv(path)

    print(f"\n{product}")
    print("File:", path)
    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df.columns))
    print("Narrative column exists:",
          "Consumer complaint narrative" in df.columns)

    assert len(df) > 0
    assert "Consumer complaint narrative" in df.columns
    assert df["Consumer complaint narrative"].fillna("").str.strip().ne("").all()

    print("\nSample narrative:")
    print(df["Consumer complaint narrative"].iloc[0][:500])

print("\nPASS: Both CFPB datasets contain non-empty complaint narratives.")


Credit card
File: /home/sagemaker-user/complaintiq/data/raw/cfpb/credit_card.csv
Rows: 18,613
Columns: 16
Narrative column exists: True

Sample narrative:
Synchrony Bank Care Credit charged me {$1000.00} in interest XXXX XXXX I was making timely payments I had no warning this raise in interest was going occur I dont quite understand how the amount is compounded to my bill of {$1500.00}. Seems excessive. I can not pay my bill now as monthly payments have gone up. I have spoken to an online rep who said to call the XXXX line during business hours and they would make new arrangements. I was told I needed to make a {$900.00} payment for the interest c

Checking or savings account
File: /home/sagemaker-user/complaintiq/data/raw/cfpb/checking_or_savings_account.csv
Rows: 19,310
Columns: 16
Narrative column exists: True

Sample narrative:
On XXXXXXXX XXXX XXXX XXXX XXXX XXXXXXXX XXXX withdrew XXXX XXXX XXXX XXXX XXXX dollars from my checking account without giving me the ten-day prior notice

In [7]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone

import requests

from src.config import ECFR_RAW, UA

BASE_URL = "https://www.ecfr.gov/api/versioner/v1"

# Regulatory parts selected for ComplaintIQ.
REGULATIONS = [
    {"title": 12, "part": "1002", "name": "Regulation B - Equal Credit Opportunity"},
    {"title": 12, "part": "1005", "name": "Regulation E - Electronic Fund Transfers"},
    {"title": 12, "part": "1006", "name": "Regulation F - Debt Collection"},
    {"title": 12, "part": "1022", "name": "Regulation V - Fair Credit Reporting"},
    {"title": 12, "part": "1026", "name": "Regulation Z - Truth in Lending"},
    {"title": 16, "part": "433", "name": "FTC Holder Rule"},
]

headers = dict(UA)
headers.setdefault("User-Agent", "ComplaintIQ-capstone/1.0")

session = requests.Session()
session.headers.update(headers)

# Discover the available publication date for each title.
response = session.get(f"{BASE_URL}/titles.json", timeout=60)
response.raise_for_status()
titles_payload = response.json()

title_metadata = {
    int(item["number"]): item
    for item in titles_payload["titles"]
}

download_manifest = []

for regulation in REGULATIONS:
    title = regulation["title"]
    part = regulation["part"]
    name = regulation["name"]

    metadata = title_metadata.get(title)
    if not metadata:
        raise RuntimeError(f"Title {title} missing from eCFR title list.")

    effective_date = (
        metadata.get("up_to_date_as_of")
        or metadata.get("latest_issue_date")
    )

    if not effective_date:
        raise RuntimeError(
            f"No available publication date found for Title {title}."
        )

    url = (
        f"{BASE_URL}/full/{effective_date}/title-{title}.xml"
    )

    print(f"Downloading Title {title}, Part {part}: {name}")
    print(f"Publication date: {effective_date}")

    response = session.get(
        url,
        params={"part": part},
        timeout=(30, 180),
    )
    response.raise_for_status()

    content = response.content

    # Validate that the returned document looks like XML.
    preview = content[:200].lstrip().lower()
    if not content or b"<" not in preview:
        raise RuntimeError(
            f"Unexpected XML response for Title {title}, Part {part}."
        )

    filename = f"title_{title}_part_{part}_{effective_date}.xml"
    destination = ECFR_RAW / filename
    destination.write_bytes(content)

    download_manifest.append({
        "title": title,
        "part": part,
        "name": name,
        "publication_date": effective_date,
        "url": response.url,
        "file": str(destination),
        "bytes": len(content),
        "sha256": __import__("hashlib").sha256(content).hexdigest(),
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    })

    print(f"Saved: {destination.name} ({len(content):,} bytes)")
    time.sleep(1)

manifest_path = ECFR_RAW / "ecfr_manifest.json"
manifest_path.write_text(
    json.dumps(download_manifest, indent=2),
    encoding="utf-8",
)

print(f"\nManifest saved: {manifest_path}")
print(f"Regulatory XML files downloaded: {len(download_manifest)}")

Publication date: 2026-11-03
Saved: title_12_part_1002_2026-11-03.xml (504,474 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1005_2026-11-03.xml (859,423 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1006_2026-11-03.xml (231,338 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1022_2026-11-03.xml (272,782 bytes)
Publication date: 2026-11-03
Saved: title_12_part_1026_2026-11-03.xml (3,946,980 bytes)
Publication date: 2026-11-03
Saved: title_16_part_433_2026-11-03.xml (6,582 bytes)

Manifest saved: /home/sagemaker-user/complaintiq/data/raw/ecfr/ecfr_manifest.json
Regulatory XML files downloaded: 6


In [8]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json

xml_files = sorted(ECFR_RAW.glob("title_*_part_*.xml"))

assert len(xml_files) == len(REGULATIONS), (
    f"Expected {len(REGULATIONS)} XML files; found {len(xml_files)}"
)

for path in xml_files:
    assert path.stat().st_size > 0, f"Empty file: {path}"

    try:
        root = ET.parse(path).getroot()
    except ET.ParseError as exc:
        raise RuntimeError(f"Invalid XML in {path.name}: {exc}") from exc

    print(
        f"PASS: {path.name} | "
        f"{path.stat().st_size:,} bytes | "
        f"Root element: {root.tag}"
    )

manifest = json.loads(
    (ECFR_RAW / "ecfr_manifest.json").read_text(encoding="utf-8")
)

assert len(manifest) == len(REGULATIONS)
print("\nPASS: eCFR files are valid XML and the manifest is present.")

PASS: title_12_part_1002_2026-11-03.xml | 504,474 bytes | Root element: DIV5
PASS: title_12_part_1005_2026-11-03.xml | 859,423 bytes | Root element: DIV5
PASS: title_12_part_1006_2026-11-03.xml | 231,338 bytes | Root element: DIV5
PASS: title_12_part_1022_2026-11-03.xml | 272,782 bytes | Root element: DIV5
PASS: title_12_part_1026_2026-11-03.xml | 3,946,980 bytes | Root element: DIV5
PASS: title_16_part_433_2026-11-03.xml | 6,582 bytes | Root element: DIV5

PASS: eCFR files are valid XML and the manifest is present.


In [9]:
from pathlib import Path
import xml.etree.ElementTree as ET
import json

xml_files = sorted(ECFR_RAW.glob("title_*_part_*.xml"))

assert len(xml_files) == len(REGULATIONS), (
    f"Expected {len(REGULATIONS)} XML files; found {len(xml_files)}"
)

for path in xml_files:
    assert path.stat().st_size > 0, f"Empty file: {path}"

    try:
        root = ET.parse(path).getroot()
    except ET.ParseError as exc:
        raise RuntimeError(f"Invalid XML in {path.name}: {exc}") from exc

    print(
        f"PASS: {path.name} | "
        f"{path.stat().st_size:,} bytes | "
        f"Root element: {root.tag}"
    )

manifest = json.loads(
    (ECFR_RAW / "ecfr_manifest.json").read_text(encoding="utf-8")
)

assert len(manifest) == len(REGULATIONS)
print("\nPASS: eCFR files are valid XML and the manifest is present.")

PASS: title_12_part_1002_2026-11-03.xml | 504,474 bytes | Root element: DIV5
PASS: title_12_part_1005_2026-11-03.xml | 859,423 bytes | Root element: DIV5
PASS: title_12_part_1006_2026-11-03.xml | 231,338 bytes | Root element: DIV5
PASS: title_12_part_1022_2026-11-03.xml | 272,782 bytes | Root element: DIV5
PASS: title_12_part_1026_2026-11-03.xml | 3,946,980 bytes | Root element: DIV5
PASS: title_16_part_433_2026-11-03.xml | 6,582 bytes | Root element: DIV5

PASS: eCFR files are valid XML and the manifest is present.


In [3]:
from pathlib import Path

PDF_DIR = PROJECT_ROOT / "data" / "raw" / "pdf"

EXPECTED_PDFS = [
    "exam_manual_v2_2012.pdf",
    "sh_issue17_2018-09.pdf",
    "sh_issue18_2019-03.pdf",
    "sh_issue22_2020-09.pdf",
    "sh_issue23_2021-01.pdf",
]

print("PDF directory:", PDF_DIR)
print("=" * 70)

for filename in EXPECTED_PDFS:
    file_path = PDF_DIR / filename

    if file_path.exists():
        print(
            f"FOUND  | {filename} | "
            f"{file_path.stat().st_size:,} bytes"
        )
    else:
        print(f"MISSING | {filename}")

print("=" * 70)

found = sum((PDF_DIR / name).exists() for name in EXPECTED_PDFS)

print(f"Found: {found}/{len(EXPECTED_PDFS)}")
print(f"Missing: {len(EXPECTED_PDFS) - found}")

PDF directory: /home/sagemaker-user/complaintiq/data/raw/pdf
MISSING | exam_manual_v2_2012.pdf
MISSING | sh_issue17_2018-09.pdf
MISSING | sh_issue18_2019-03.pdf
MISSING | sh_issue22_2020-09.pdf
MISSING | sh_issue23_2021-01.pdf
Found: 0/5
Missing: 5


In [9]:
from pathlib import Path

PDF_DIR = PROJECT_ROOT / "data" / "raw" / "pdf"
PDF_DIR.mkdir(parents=True, exist_ok=True)

PDF_SOURCES = {
    "exam_manual_v2_2012.pdf": (
        "https://files.consumerfinance.gov/f/201210_cfpb_supervision-and-examination-manual-v2.pdf"
    ),
    "sh_issue17_2018-09.pdf": (
        "https://www.consumerfinance.gov/documents/6817/bcfp_supervisory-highlights_issue-17_2018-09.pdf"
    ),
    "sh_issue18_2019-03.pdf": (
        "https://files.consumerfinance.gov/f/documents/cfpb_supervisory-highlights_issue-18_032019.pdf"
    ),
    "sh_issue22_2020-09.pdf": (
        "https://www.consumerfinance.gov/documents/9164/cfpb_supervisory-highlights_issue-22_2020-09.pdf"
    ),
    "sh_issue23_2021-01.pdf": (
        "https://files.consumerfinance.gov/f/documents/cfpb-supervisory-highlights_issue-23_2021-01.pdf"
    ),
}

def is_valid_pdf(path):
    if not path.exists() or path.stat().st_size < 1000:
        return False
    with path.open("rb") as f:
        return f.read(5) == b"%PDF-"

for filename in PDF_SOURCES:
    path = PDF_DIR / filename
    if is_valid_pdf(path):
        print(f"VALID   | {filename} | {path.stat().st_size:,} bytes")
    elif path.exists():
        print(f"INVALID | {filename} | exists but is not a valid PDF")
    else:
        print(f"MISSING | {filename}")

print(f"\nPDF directory: {PDF_DIR}")

VALID   | exam_manual_v2_2012.pdf | 7,486,372 bytes
VALID   | sh_issue17_2018-09.pdf | 195,546 bytes
VALID   | sh_issue18_2019-03.pdf | 399,852 bytes
VALID   | sh_issue22_2020-09.pdf | 463,021 bytes
VALID   | sh_issue23_2021-01.pdf | 774,257 bytes

PDF directory: /home/sagemaker-user/complaintiq/data/raw/pdf


In [5]:
import requests
import time

session = requests.Session()
session.headers.update({
    "User-Agent": "ComplaintIQ Capstone Research Project/1.0 (joshua.paulv@ltm.com)"
})

download_results = []

for filename, url in PDF_SOURCES.items():
    destination = PDF_DIR / filename

    if is_valid_pdf(destination):
        print(f"SKIP    | Already valid: {filename}")
        download_results.append({
            "filename": filename,
            "status": "REUSED",
            "bytes": destination.stat().st_size,
            "url": url,
        })
        continue

    print(f"DOWNLOAD | {filename}")

    try:
        response = session.get(url, timeout=(20, 180))
        response.raise_for_status()

        content = response.content

        if len(content) < 1000 or not content.startswith(b"%PDF-"):
            raise ValueError(
                f"Response is not a valid PDF. "
                f"Content-Type={response.headers.get('Content-Type')}; "
                f"bytes={len(content)}"
            )

        temporary = destination.with_suffix(".pdf.part")
        temporary.write_bytes(content)

        # Validate temporary file before replacing destination.
        if not is_valid_pdf(temporary):
            temporary.unlink(missing_ok=True)
            raise ValueError("Downloaded file failed PDF signature validation.")

        temporary.replace(destination)

        print(f"SUCCESS  | {filename} | {len(content):,} bytes")

        download_results.append({
            "filename": filename,
            "status": "DOWNLOADED",
            "bytes": len(content),
            "url": url,
        })

        time.sleep(1)

    except Exception as exc:
        print(f"FAILED   | {filename} | {type(exc).__name__}: {exc}")
        download_results.append({
            "filename": filename,
            "status": "FAILED",
            "error": str(exc),
            "url": url,
        })

print("\nDownload summary")
for item in download_results:
    print(item["status"], "|", item["filename"])

failed = [item for item in download_results if item["status"] == "FAILED"]

if failed:
    print(f"\nSTOP: {len(failed)} PDF(s) failed. Do not mark Phase 1 complete.")
else:
    print("\nAll five PDF files are available.")

DOWNLOAD | exam_manual_v2_2012.pdf
SUCCESS  | exam_manual_v2_2012.pdf | 7,486,372 bytes
DOWNLOAD | sh_issue17_2018-09.pdf
SUCCESS  | sh_issue17_2018-09.pdf | 195,546 bytes
DOWNLOAD | sh_issue18_2019-03.pdf
SUCCESS  | sh_issue18_2019-03.pdf | 399,852 bytes
DOWNLOAD | sh_issue22_2020-09.pdf
SUCCESS  | sh_issue22_2020-09.pdf | 463,021 bytes
DOWNLOAD | sh_issue23_2021-01.pdf
FAILED   | sh_issue23_2021-01.pdf | HTTPError: 404 Client Error: Not Found for url: https://files.consumerfinance.gov/f/documents/cfpb-supervisory-highlights_issue-23_2021-01.pdf

Download summary
DOWNLOADED | exam_manual_v2_2012.pdf
DOWNLOADED | sh_issue17_2018-09.pdf
DOWNLOADED | sh_issue18_2019-03.pdf
DOWNLOADED | sh_issue22_2020-09.pdf
FAILED | sh_issue23_2021-01.pdf

STOP: 1 PDF(s) failed. Do not mark Phase 1 complete.


In [10]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from pypdf import PdfReader

PDF_DIR = Path(
    "/home/sagemaker-user/complaintiq/data/raw/pdf"
)

PDF_FILES = [
    "exam_manual_v2_2012.pdf",
    "sh_issue17_2018-09.pdf",
    "sh_issue18_2019-03.pdf",
    "sh_issue22_2020-09.pdf",
    "sh_issue23_2021-01.pdf",
]

print("=" * 70)
print("COMPLAINTIQ — PDF STRUCTURAL VALIDATION")
print("=" * 70)

pdf_validation_passed = True

for filename in PDF_FILES:
    path = PDF_DIR / filename

    try:
        if not path.exists():
            raise FileNotFoundError(filename)

        reader = PdfReader(str(path))
        page_count = len(reader.pages)

        assert page_count > 0, "PDF contains no pages"

        print(
            f"PASS | {filename} | "
            f"{page_count} pages | "
            f"{path.stat().st_size:,} bytes"
        )

    except Exception as e:
        pdf_validation_passed = False
        print(f"FAIL | {filename} | {type(e).__name__}: {e}")

print("-" * 70)

if pdf_validation_passed:
    print("SUCCESS — All five PDFs are readable.")
else:
    print("STOP — One or more PDFs failed validation.")

COMPLAINTIQ — PDF STRUCTURAL VALIDATION
PASS | exam_manual_v2_2012.pdf | 924 pages | 7,486,372 bytes
PASS | sh_issue17_2018-09.pdf | 23 pages | 195,546 bytes


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 99 0 (offset 0)
Ignoring wrong pointing object 105 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignoring wrong pointing object 110 0 (offset 0)


PASS | sh_issue18_2019-03.pdf | 21 pages | 399,852 bytes
PASS | sh_issue22_2020-09.pdf | 38 pages | 463,021 bytes
PASS | sh_issue23_2021-01.pdf | 28 pages | 774,257 bytes
----------------------------------------------------------------------
SUCCESS — All five PDFs are readable.


In [4]:
from pathlib import Path
import json

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

ECFR_DIR = PROJECT_ROOT / "data" / "raw" / "ecfr"

print("=" * 70)
print("COMPLAINTIQ — eCFR DIRECTORY INSPECTION")
print("=" * 70)

print("\neCFR directory:", ECFR_DIR)
print("Directory exists:", ECFR_DIR.exists())

if ECFR_DIR.exists():
    print("\nFiles and directories:")

    for item in sorted(ECFR_DIR.iterdir()):
        if item.is_file():
            print(f"FILE | {item.name} | {item.stat().st_size:,} bytes")
        else:
            print(f"DIR  | {item.name}")

print("\n" + "=" * 70)
print("EXISTING eCFR MANIFEST")
print("=" * 70)

ecfr_manifest_path = ECFR_DIR / "ecfr_manifest.json"

if ecfr_manifest_path.exists():
    ecfr_data = json.loads(
        ecfr_manifest_path.read_text(encoding="utf-8")
    )

    print("Manifest exists:", True)
    print("Data type:", type(ecfr_data).__name__)

    preview = (
        ecfr_data[:2]
        if isinstance(ecfr_data, list)
        else ecfr_data
    )

    print(json.dumps(preview, indent=2)[:6000])

else:
    print("No ecfr_manifest.json found.")

COMPLAINTIQ — eCFR DIRECTORY INSPECTION

eCFR directory: /home/sagemaker-user/complaintiq/data/raw/ecfr
Directory exists: True

Files and directories:
FILE | ecfr_manifest.json | 2,889 bytes
FILE | title_12_part_1002_2026-11-03.xml | 504,474 bytes
FILE | title_12_part_1005_2026-11-03.xml | 859,423 bytes
FILE | title_12_part_1006_2026-11-03.xml | 231,338 bytes
FILE | title_12_part_1022_2026-11-03.xml | 272,782 bytes
FILE | title_12_part_1026_2026-11-03.xml | 3,946,980 bytes
FILE | title_16_part_433_2026-11-03.xml | 6,582 bytes

EXISTING eCFR MANIFEST
Manifest exists: True
Data type: list
[
  {
    "title": 12,
    "part": "1002",
    "name": "Regulation B - Equal Credit Opportunity",
    "publication_date": "2026-11-03",
    "url": "https://www.ecfr.gov/api/versioner/v1/full/2026-11-03/title-12.xml?part=1002",
    "file": "/home/sagemaker-user/complaintiq/data/raw/ecfr/title_12_part_1002_2026-11-03.xml",
    "bytes": 504474,
    "sha256": "e05a7a592ffaf9b5519222a2aeba5f3ea34908db52edba8

In [5]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import xml.etree.ElementTree as ET
import csv

from pypdf import PdfReader

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")
DATA_DIR = PROJECT_ROOT / "data"

CFPB_DIR = DATA_DIR / "raw" / "cfpb"
ECFR_DIR = DATA_DIR / "raw" / "ecfr"
PDF_DIR = DATA_DIR / "raw" / "pdf"

ECFR_MANIFEST_PATH = ECFR_DIR / "ecfr_manifest.json"
FINAL_MANIFEST_PATH = DATA_DIR / "manifest.json"

# ============================================================
# 2. EXPECTED INPUT FILES
# ============================================================

CFPB_FILES = [
    "credit_card.csv",
    "checking_or_savings_account.csv",
]

ECFR_FILES = [
    "title_12_part_1002_2026-11-03.xml",
    "title_12_part_1005_2026-11-03.xml",
    "title_12_part_1006_2026-11-03.xml",
    "title_12_part_1022_2026-11-03.xml",
    "title_12_part_1026_2026-11-03.xml",
    "title_16_part_433_2026-11-03.xml",
]

PDF_FILES = [
    "exam_manual_v2_2012.pdf",
    "sh_issue17_2018-09.pdf",
    "sh_issue18_2019-03.pdf",
    "sh_issue22_2020-09.pdf",
    "sh_issue23_2021-01.pdf",
]

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def calculate_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def file_metadata(path):
    return {
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "bytes": path.stat().st_size,
        "sha256": calculate_sha256(path),
    }


def check_file_exists(path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")


# ============================================================
# 4. VALIDATE CFPB CSV FILES
# ============================================================

print("=" * 75)
print("PHASE 1 — CFPB DATA VALIDATION")
print("=" * 75)

cfpb_entries = []

for filename in CFPB_FILES:

    path = CFPB_DIR / filename
    check_file_exists(path)

    with open(path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)

        columns = reader.fieldnames or []

        if not columns:
            raise ValueError(f"No CSV headers found: {filename}")

        narrative_columns = [
            column for column in columns
            if "narrative" in column.lower()
        ]

        if not narrative_columns:
            raise ValueError(
                f"Narrative column missing in {filename}. "
                f"Available columns: {columns}"
            )

        row_count = 0
        narrative_count = 0

        for row in reader:
            row_count += 1

            if any(
                (row.get(column) or "").strip()
                for column in narrative_columns
            ):
                narrative_count += 1

    if row_count == 0:
        raise ValueError(f"CSV contains no records: {filename}")

    entry = file_metadata(path)
    entry.update({
        "type": "cfpb_complaints",
        "records": row_count,
        "narrative_records": narrative_count,
        "narrative_columns": narrative_columns,
    })

    cfpb_entries.append(entry)

    print(
        f"PASS | {filename} | "
        f"{row_count:,} records | "
        f"{narrative_count:,} with narratives"
    )


# ============================================================
# 5. VALIDATE eCFR XML FILES
# ============================================================

print("\n" + "=" * 75)
print("PHASE 1 — eCFR XML VALIDATION")
print("=" * 75)

if not ECFR_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"eCFR manifest missing: {ECFR_MANIFEST_PATH}"
    )

ecfr_source_manifest = json.loads(
    ECFR_MANIFEST_PATH.read_text(encoding="utf-8")
)

if not isinstance(ecfr_source_manifest, list):
    raise ValueError("Unexpected eCFR manifest format.")

if len(ecfr_source_manifest) != 6:
    raise ValueError(
        f"Expected 6 eCFR manifest entries, "
        f"found {len(ecfr_source_manifest)}"
    )

ecfr_entries = []

for filename in ECFR_FILES:

    path = ECFR_DIR / filename
    check_file_exists(path)

    # Parse XML to confirm structural readability.
    tree = ET.parse(path)
    root = tree.getroot()

    # Match the actual source metadata using the filename.
    source_entry = next(
        (
            item for item in ecfr_source_manifest
            if Path(item.get("file", "")).name == filename
        ),
        None
    )

    if source_entry is None:
        raise ValueError(
            f"No matching eCFR manifest entry for {filename}"
        )

    actual_hash = calculate_sha256(path)

    if source_entry.get("sha256") != actual_hash:
        raise ValueError(
            f"SHA256 mismatch for {filename}"
        )

    entry = file_metadata(path)

    entry.update({
        "type": "ecfr_regulation",
        "title": source_entry.get("title"),
        "part": source_entry.get("part"),
        "name": source_entry.get("name"),
        "publication_date": source_entry.get("publication_date"),
        "source_url": source_entry.get("url"),
        "root_element": root.tag,
    })

    ecfr_entries.append(entry)

    print(
        f"PASS | {filename} | "
        f"{path.stat().st_size:,} bytes | "
        f"XML parsed successfully"
    )


# ============================================================
# 6. VALIDATE PDF FILES
# ============================================================

print("\n" + "=" * 75)
print("PHASE 1 — PDF VALIDATION")
print("=" * 75)

pdf_entries = []

for filename in PDF_FILES:

    path = PDF_DIR / filename
    check_file_exists(path)

    if path.stat().st_size < 1000:
        raise ValueError(f"PDF unexpectedly small: {filename}")

    with open(path, "rb") as file:
        if file.read(5) != b"%PDF-":
            raise ValueError(f"Invalid PDF signature: {filename}")

    reader = PdfReader(str(path))
    page_count = len(reader.pages)

    if page_count == 0:
        raise ValueError(f"PDF has no pages: {filename}")

    entry = file_metadata(path)
    entry.update({
        "type": "reference_pdf",
        "pages": page_count,
    })

    pdf_entries.append(entry)

    print(
        f"PASS | {filename} | "
        f"{page_count} pages | "
        f"{path.stat().st_size:,} bytes"
    )


# ============================================================
# 7. BUILD CONSOLIDATED MANIFEST
# ============================================================

print("\n" + "=" * 75)
print("PHASE 1 — BUILDING CONSOLIDATED MANIFEST")
print("=" * 75)

all_entries = cfpb_entries + ecfr_entries + pdf_entries

manifest = {
    "project": "ComplaintIQ",
    "phase": 1,
    "manifest_version": "1.0",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "summary": {
        "cfpb_csv_files": len(cfpb_entries),
        "ecfr_xml_files": len(ecfr_entries),
        "reference_pdf_files": len(pdf_entries),
        "total_files": len(all_entries),
        "total_cfpb_records": sum(
            item["records"] for item in cfpb_entries
        ),
        "total_cfpb_narrative_records": sum(
            item["narrative_records"] for item in cfpb_entries
        ),
        "total_pdf_pages": sum(
            item["pages"] for item in pdf_entries
        ),
    },
    "files": all_entries,
}

# Preserve an existing manifest before replacing it.
if FINAL_MANIFEST_PATH.exists():

    backup_path = FINAL_MANIFEST_PATH.with_name(
        "manifest.json.previous"
    )

    if backup_path.exists():
        timestamp = datetime.now(timezone.utc).strftime(
            "%Y%m%dT%H%M%SZ"
        )
        backup_path = FINAL_MANIFEST_PATH.with_name(
            f"manifest_{timestamp}.json.previous"
        )

    backup_path.write_bytes(FINAL_MANIFEST_PATH.read_bytes())

    print("Previous manifest backed up to:", backup_path)

DATA_DIR.mkdir(parents=True, exist_ok=True)

temporary_manifest = DATA_DIR / "manifest.json.tmp"

temporary_manifest.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Validate the generated JSON before publishing it.
json.loads(temporary_manifest.read_text(encoding="utf-8"))

temporary_manifest.replace(FINAL_MANIFEST_PATH)

print("Manifest created:", FINAL_MANIFEST_PATH)


# ============================================================
# 8. FINAL PHASE 1 SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("COMPLAINTIQ — PHASE 1 SUMMARY")
print("=" * 75)

print(json.dumps(manifest["summary"], indent=2))

print("\nManifest size:",
      f"{FINAL_MANIFEST_PATH.stat().st_size:,} bytes")

print("\nManifest SHA256:",
      calculate_sha256(FINAL_MANIFEST_PATH))

print("\n" + "=" * 75)
print("PHASE 1 VALIDATION COMPLETED SUCCESSFULLY")
print("=" * 75)


PHASE 1 — CFPB DATA VALIDATION
PASS | credit_card.csv | 18,613 records | 18,613 with narratives
PASS | checking_or_savings_account.csv | 19,310 records | 19,310 with narratives

PHASE 1 — eCFR XML VALIDATION
PASS | title_12_part_1002_2026-11-03.xml | 504,474 bytes | XML parsed successfully
PASS | title_12_part_1005_2026-11-03.xml | 859,423 bytes | XML parsed successfully
PASS | title_12_part_1006_2026-11-03.xml | 231,338 bytes | XML parsed successfully
PASS | title_12_part_1022_2026-11-03.xml | 272,782 bytes | XML parsed successfully
PASS | title_12_part_1026_2026-11-03.xml | 3,946,980 bytes | XML parsed successfully
PASS | title_16_part_433_2026-11-03.xml | 6,582 bytes | XML parsed successfully

PHASE 1 — PDF VALIDATION


Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 99 0 (offset 0)
Ignoring wrong pointing object 105 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignoring wrong pointing object 110 0 (offset 0)


PASS | exam_manual_v2_2012.pdf | 924 pages | 7,486,372 bytes
PASS | sh_issue17_2018-09.pdf | 23 pages | 195,546 bytes
PASS | sh_issue18_2019-03.pdf | 21 pages | 399,852 bytes
PASS | sh_issue22_2020-09.pdf | 38 pages | 463,021 bytes
PASS | sh_issue23_2021-01.pdf | 28 pages | 774,257 bytes

PHASE 1 — BUILDING CONSOLIDATED MANIFEST
Manifest created: /home/sagemaker-user/complaintiq/data/manifest.json

COMPLAINTIQ — PHASE 1 SUMMARY
{
  "cfpb_csv_files": 2,
  "ecfr_xml_files": 6,
  "reference_pdf_files": 5,
  "total_files": 13,
  "total_cfpb_records": 37923,
  "total_cfpb_narrative_records": 37923,
  "total_pdf_pages": 1034
}

Manifest size: 5,850 bytes

Manifest SHA256: 56c796f38f4d70845113cb48fe279b388aa54b33fb881f2e00f0c3a1ffc036d0

PHASE 1 VALIDATION COMPLETED SUCCESSFULLY


In [6]:

import json
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")
manifest_path = PROJECT_ROOT / "data" / "manifest.json"

assert manifest_path.exists(), "Manifest does not exist."

data = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

assert data["project"] == "ComplaintIQ"
assert data["phase"] == 1

summary = data["summary"]

assert summary["cfpb_csv_files"] == 2
assert summary["ecfr_xml_files"] == 6
assert summary["reference_pdf_files"] == 5
assert summary["total_files"] == 13

assert len(data["files"]) == 13

print("=" * 65)
print("COMPLAINTIQ — PHASE 1 ACCEPTANCE CHECK")
print("=" * 65)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nPASS — Consolidated manifest is valid.")
print("PASS — All 13 expected source files are represented.")
print("PASS — Phase 1 acceptance checks completed.")


COMPLAINTIQ — PHASE 1 ACCEPTANCE CHECK
cfpb_csv_files: 2
ecfr_xml_files: 6
reference_pdf_files: 5
total_files: 13
total_cfpb_records: 37923
total_cfpb_narrative_records: 37923
total_pdf_pages: 1034

PASS — Consolidated manifest is valid.
PASS — All 13 expected source files are represented.
PASS — Phase 1 acceptance checks completed.


In [7]:
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

CHECK_PATHS = [
    "data/complaintiq.duckdb",

    "data/processed/cfpb_index_pool_jan_may_2026.csv",
    "data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv",
    "data/processed/cfpb_holdout_june_2026.csv",

    "data/processed/reg_sections.jsonl",
    "data/processed/pdf_pages.jsonl",
    "data/processed/document_chunks.jsonl",
]

print("=" * 75)
print("COMPLAINTIQ — PHASE 2 EXISTING OUTPUT INSPECTION")
print("=" * 75)

for relative_path in CHECK_PATHS:

    path = PROJECT_ROOT / relative_path

    if path.exists():
        print(
            f"EXISTS | {relative_path} | "
            f"{path.stat().st_size:,} bytes"
        )
    else:
        print(f"MISSING | {relative_path}")

print("\n" + "=" * 75)
print("PHASE 2 INPUT FILE COUNTS")
print("=" * 75)

for folder, pattern in [
    ("data/raw/cfpb", "*.csv"),
    ("data/raw/ecfr", "*.xml"),
    ("data/raw/pdf", "*.pdf"),
]:
    directory = PROJECT_ROOT / folder
    files = sorted(directory.glob(pattern))

    print(f"{folder}: {len(files)} files")

COMPLAINTIQ — PHASE 2 EXISTING OUTPUT INSPECTION
MISSING | data/complaintiq.duckdb
MISSING | data/processed/cfpb_index_pool_jan_may_2026.csv
MISSING | data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv
MISSING | data/processed/cfpb_holdout_june_2026.csv
MISSING | data/processed/reg_sections.jsonl
MISSING | data/processed/pdf_pages.jsonl
MISSING | data/processed/document_chunks.jsonl

PHASE 2 INPUT FILE COUNTS
data/raw/cfpb: 2 files
data/raw/ecfr: 6 files
data/raw/pdf: 5 files


In [8]:
from pathlib import Path
import zipfile

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

ARCHIVE_DIR = PROJECT_ROOT / "data" / "raw" / "cfpb_archive"

EXPECTED_ARCHIVES = [
    "january_february_2026.zip",
    "march_2026.zip",
    "april_2026.zip",
    "may_2026.zip",
    "june_2026.zip",
]

print("=" * 75)
print("COMPLAINTIQ — CFPB ARCHIVE INSPECTION")
print("=" * 75)

print("Archive directory:", ARCHIVE_DIR)
print("Directory exists:", ARCHIVE_DIR.exists())

if not ARCHIVE_DIR.exists():
    print("\nSTOP — CFPB archive directory is missing.")
else:
    for filename in EXPECTED_ARCHIVES:
        path = ARCHIVE_DIR / filename

        if not path.exists():
            print(f"MISSING | {filename}")
            continue

        print(f"\nFOUND | {filename}")
        print(f"Size: {path.stat().st_size:,} bytes")

        try:
            with zipfile.ZipFile(path, "r") as zf:
                bad_file = zf.testzip()

                if bad_file:
                    print("INVALID ZIP | Corrupt member:", bad_file)
                    continue

                csv_files = [
                    name for name in zf.namelist()
                    if name.lower().endswith(".csv")
                ]

                print("ZIP integrity: PASS")
                print("CSV files:", len(csv_files))

                for csv_name in csv_files:
                    info = zf.getinfo(csv_name)
                    print(
                        f"  {csv_name} | "
                        f"{info.file_size:,} bytes"
                    )

        except zipfile.BadZipFile:
            print("INVALID ZIP | Archive is corrupt")

print("\n" + "=" * 75)
print("ARCHIVE INSPECTION COMPLETED")
print("=" * 75)

COMPLAINTIQ — CFPB ARCHIVE INSPECTION
Archive directory: /home/sagemaker-user/complaintiq/data/raw/cfpb_archive
Directory exists: True

FOUND | january_february_2026.zip
Size: 36,832,036 bytes
ZIP integrity: PASS
CSV files: 1
  CCDB_Export_15_January_2026_through_February_2026.csv | 366,507,123 bytes

FOUND | march_2026.zip
Size: 22,053,643 bytes
ZIP integrity: PASS
CSV files: 1
  CCDB_Export_16_March_2026.csv | 219,594,182 bytes

FOUND | april_2026.zip
Size: 22,963,936 bytes
ZIP integrity: PASS
CSV files: 1
  CCDB_Export_17_April_2026.csv | 223,480,946 bytes

FOUND | may_2026.zip
Size: 21,033,928 bytes
ZIP integrity: PASS
CSV files: 1
  CCDB_Export_18_May_2026.csv | 219,622,001 bytes

FOUND | june_2026.zip
Size: 19,624,767 bytes
ZIP integrity: PASS
CSV files: 1
  CCDB_Export_19_June_2026.csv | 219,864,163 bytes

ARCHIVE INSPECTION COMPLETED


In [11]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "cfpb"

DB_PATH = DATA_DIR / "complaintiq.duckdb"

DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DIR.glob("*.csv"))

print("CFPB CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name, f"({file.stat().st_size:,} bytes)")

if len(csv_files) != 2:
    raise RuntimeError(
        f"Expected 2 CFPB CSV files, found {len(csv_files)}"
    )

# Create DuckDB database
con = duckdb.connect(str(DB_PATH))

print("\nDuckDB database:", DB_PATH)

# Load raw CFPB CSV files
csv_pattern = (RAW_DIR / "*.csv").as_posix()

con.execute(f"""
    CREATE OR REPLACE TABLE complaints_raw AS
    SELECT *
    FROM read_csv_auto(
        '{csv_pattern}',
        header = true,
        normalize_names = true,
        all_varchar = true
    )
""")

# Display table structure
print("\n=== COMPLAINTS_RAW SCHEMA ===")

schema = con.execute("""
    DESCRIBE complaints_raw
""").df()

display(schema)

# Display record count
row_count = con.execute("""
    SELECT COUNT(*)
    FROM complaints_raw
""").fetchone()[0]

print("\nTotal raw complaint records:", f"{row_count:,}")

# Display sample records
print("\n=== SAMPLE RECORDS ===")

display(
    con.execute("""
        SELECT *
        FROM complaints_raw
        LIMIT 3
    """).df()
)

con.close()

print("\nPASS — DuckDB database created and raw CFPB data loaded.")

CFPB CSV files found: 2
checking_or_savings_account.csv (32,046,860 bytes)
credit_card.csv (33,552,641 bytes)

DuckDB database: /home/sagemaker-user/complaintiq/data/complaintiq.duckdb

=== COMPLAINTS_RAW SCHEMA ===


,column_name,column_type,null,key,default,extra
0,date_received,VARCHAR,YES,None,None,None
1,product,VARCHAR,YES,None,None,None
2,subproduct,VARCHAR,YES,None,None,None
3,issue,VARCHAR,YES,None,None,None
4,subissue,VARCHAR,YES,None,None,None
5,consumer_complaint_narrative,VARCHAR,YES,None,None,None
6,company_public_response,VARCHAR,YES,None,None,None
7,company,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,zip_code,VARCHAR,YES,None,None,None



Total raw complaint records: 37,923

=== SAMPLE RECORDS ===


,date_received,product,subproduct,issue,subissue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id
0,2026-01-23,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,On XXXXXXXX XXXX XXXX XXXX XXXX XXXXXXXX XXXX ...,None,FIFTH THIRD FINANCIAL CORPORATION,IN,464XX,None,Web,2026-07-01,Closed with explanation,Yes,18962200
1,2026-02-22,Checking or savings account,Checking account,Closing an account,Company closed your account,Cashapp closed my account and didnt provide a ...,None,"Block, Inc.",IL,60563,None,Web,2026-02-22,Closed with explanation,Yes,19687242
2,2026-02-22,Checking or savings account,Checking account,Managing an account,Deposits and withdrawals,"I am in the process of suing HSBC US, please m...",None,HSBC NORTH AMERICA HOLDINGS INC.,TX,XXXXX,None,Web,2026-02-22,Closed with explanation,Yes,19687176



PASS — DuckDB database created and raw CFPB data loaded.


In [12]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")
DB_PATH = PROJECT_ROOT / "data" / "complaintiq.duckdb"

con = duckdb.connect(str(DB_PATH))

print("=== TABLES ===")

display(
    con.execute("""
        SHOW TABLES
    """).df()
)

print("\n=== COMPLAINTS_RAW SCHEMA ===")

display(
    con.execute("""
        DESCRIBE complaints_raw
    """).df()
)

print("\n=== SAMPLE DATA ===")

display(
    con.execute("""
        SELECT *
        FROM complaints_raw
        LIMIT 2
    """).df()
)

con.close()

=== TABLES ===


,name
0,complaints_raw



=== COMPLAINTS_RAW SCHEMA ===


,column_name,column_type,null,key,default,extra
0,date_received,VARCHAR,YES,None,None,None
1,product,VARCHAR,YES,None,None,None
2,subproduct,VARCHAR,YES,None,None,None
3,issue,VARCHAR,YES,None,None,None
4,subissue,VARCHAR,YES,None,None,None
5,consumer_complaint_narrative,VARCHAR,YES,None,None,None
6,company_public_response,VARCHAR,YES,None,None,None
7,company,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,zip_code,VARCHAR,YES,None,None,None



=== SAMPLE DATA ===


,date_received,product,subproduct,issue,subissue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id
0,2026-01-23,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,On XXXXXXXX XXXX XXXX XXXX XXXX XXXXXXXX XXXX ...,None,FIFTH THIRD FINANCIAL CORPORATION,IN,464XX,None,Web,2026-07-01,Closed with explanation,Yes,18962200
1,2026-02-22,Checking or savings account,Checking account,Closing an account,Company closed your account,Cashapp closed my account and didnt provide a ...,None,"Block, Inc.",IL,60563,None,Web,2026-02-22,Closed with explanation,Yes,19687242


In [13]:
import duckdb
from pathlib import Path

PROJECT = Path("/home/sagemaker-user/complaintiq")
DB_PATH = PROJECT / "data/complaintiq.duckdb"

con = duckdb.connect(str(DB_PATH))

con.execute("""
CREATE OR REPLACE VIEW v_complaints AS
SELECT
    TRY_CAST(date_received AS DATE) AS date_received,
    product,
    subproduct,
    issue,
    subissue,
    consumer_complaint_narrative AS narrative,
    company_public_response,
    company,
    state,
    zip_code,
    tags,
    submitted_via,
    TRY_CAST(date_sent_to_company AS DATE) AS date_sent_to_company,
    company_response_to_consumer,
    timely_response,
    TRY_CAST(complaint_id AS BIGINT) AS complaint_id
FROM complaints_raw
""")

print("=== VIEW CREATED ===")

display(con.execute("DESCRIBE v_complaints").df())

print("\n=== DATA SUMMARY ===")

display(con.execute("""
SELECT
    product,
    COUNT(*) AS total_records,
    COUNT(narrative) AS records_with_narrative,
    COUNT(DISTINCT complaint_id) AS unique_complaints,
    MIN(date_received) AS earliest_date,
    MAX(date_received) AS latest_date
FROM v_complaints
GROUP BY product
ORDER BY product
""").df())

con.close()

=== VIEW CREATED ===


,column_name,column_type,null,key,default,extra
0,date_received,DATE,YES,None,None,None
1,product,VARCHAR,YES,None,None,None
2,subproduct,VARCHAR,YES,None,None,None
3,issue,VARCHAR,YES,None,None,None
4,subissue,VARCHAR,YES,None,None,None
5,narrative,VARCHAR,YES,None,None,None
6,company_public_response,VARCHAR,YES,None,None,None
7,company,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,zip_code,VARCHAR,YES,None,None,None



=== DATA SUMMARY ===


,product,total_records,records_with_narrative,unique_complaints,earliest_date,latest_date
0,Checking or savings account,19310,19310,19310,2026-01-01,2026-06-30
1,Credit card,18613,18613,18613,2026-01-01,2026-06-30


In [14]:

from pathlib import Path
import zipfile
import pandas as pd

PROJECT = Path("/home/sagemaker-user/complaintiq")
ARCHIVE_DIR = PROJECT / "data/raw/cfpb_archive"
OUTPUT_DIR = PROJECT / "data/processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_FILE = OUTPUT_DIR / "cfpb_index_pool_jan_may_2026.csv"
HOLDOUT_FILE = OUTPUT_DIR / "cfpb_holdout_june_2026.csv"

ARCHIVES = [
    "january_february_2026.zip",
    "march_2026.zip",
    "april_2026.zip",
    "may_2026.zip",
    "june_2026.zip",
]

TARGET_PRODUCTS = {
    "credit card",
    "checking or savings account",
}

DATE_COL = "Date received"
PRODUCT_COL = "Product"
NARRATIVE_COL = "Consumer complaint narrative"
ID_COL = "Complaint ID"

# Remove previous outputs so reruns start cleanly.
INDEX_FILE.unlink(missing_ok=True)
HOLDOUT_FILE.unlink(missing_ok=True)

index_rows = 0
holdout_rows = 0
index_header_written = False
holdout_header_written = False

for archive_name in ARCHIVES:
    archive_path = ARCHIVE_DIR / archive_name

    if not archive_path.exists():
        raise FileNotFoundError(archive_path)

    print(f"\nProcessing: {archive_name}")

    with zipfile.ZipFile(archive_path) as zf:
        csv_names = [
            name for name in zf.namelist()
            if name.lower().endswith(".csv")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"Expected one CSV in {archive_name}; found {len(csv_names)}"
            )

        csv_name = csv_names[0]

        with zf.open(csv_name) as raw:
            for chunk in pd.read_csv(
                raw,
                dtype=str,
                encoding="utf-8-sig",
                keep_default_na=False,
                chunksize=50000,
                low_memory=False,
            ):
                required = {
                    DATE_COL, PRODUCT_COL, NARRATIVE_COL, ID_COL
                }
                missing = required - set(chunk.columns)

                if missing:
                    raise ValueError(
                        f"{csv_name} is missing columns: {sorted(missing)}"
                    )

                dates = pd.to_datetime(
                    chunk[DATE_COL],
                    errors="coerce"
                )

                products = chunk[PRODUCT_COL].str.strip().str.lower()
                narratives = chunk[NARRATIVE_COL].str.strip()
                ids = chunk[ID_COL].str.strip()

                eligible = (
                    products.isin(TARGET_PRODUCTS)
                    & narratives.ne("")
                    & ids.ne("")
                    & dates.notna()
                )

                chunk = chunk.loc[eligible].copy()
                dates = dates.loc[eligible]

                index_mask = (
                    (dates >= "2026-01-01")
                    & (dates < "2026-06-01")
                )
                holdout_mask = (
                    (dates >= "2026-06-01")
                    & (dates < "2026-07-01")
                )

                index_chunk = chunk.loc[index_mask]
                holdout_chunk = chunk.loc[holdout_mask]

                if not index_chunk.empty:
                    index_chunk.to_csv(
                        INDEX_FILE,
                        mode="a",
                        index=False,
                        header=not index_header_written,
                        encoding="utf-8",
                    )
                    index_header_written = True
                    index_rows += len(index_chunk)

                if not holdout_chunk.empty:
                    holdout_chunk.to_csv(
                        HOLDOUT_FILE,
                        mode="a",
                        index=False,
                        header=not holdout_header_written,
                        encoding="utf-8",
                    )
                    holdout_header_written = True
                    holdout_rows += len(holdout_chunk)

    print(
        f"Index rows so far: {index_rows:,} | "
        f"June rows so far: {holdout_rows:,}"
    )

if not index_header_written or not holdout_header_written:
    raise RuntimeError("One or both output datasets are empty.")

print("\n=== CFPB DATASET GENERATION COMPLETE ===")
print("January-May index rows:", f"{index_rows:,}")
print("June holdout rows:", f"{holdout_rows:,}")
print("Index file:", INDEX_FILE)
print("Holdout file:", HOLDOUT_FILE)



Processing: january_february_2026.zip
Index rows so far: 11,542 | June rows so far: 0

Processing: march_2026.zip
Index rows so far: 18,354 | June rows so far: 0

Processing: april_2026.zip
Index rows so far: 25,734 | June rows so far: 0

Processing: may_2026.zip
Index rows so far: 31,813 | June rows so far: 0

Processing: june_2026.zip
Index rows so far: 31,813 | June rows so far: 6,110

=== CFPB DATASET GENERATION COMPLETE ===
January-May index rows: 31,813
June holdout rows: 6,110
Index file: /home/sagemaker-user/complaintiq/data/processed/cfpb_index_pool_jan_may_2026.csv
Holdout file: /home/sagemaker-user/complaintiq/data/processed/cfpb_holdout_june_2026.csv


In [15]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/home/sagemaker-user/complaintiq")
PROC = PROJECT / "data/processed"

INDEX_FILE = PROC / "cfpb_index_pool_jan_may_2026.csv"
HOLDOUT_FILE = PROC / "cfpb_holdout_june_2026.csv"

for label, path in [
    ("January-May index pool", INDEX_FILE),
    ("June holdout", HOLDOUT_FILE),
]:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path, dtype=str, keep_default_na=False)

    dates = pd.to_datetime(df["Date received"], errors="coerce")

    print(f"\n=== {label} ===")
    print("Rows:", f"{len(df):,}")
    print("Unique complaint IDs:", f"{df['Complaint ID'].nunique():,}")
    print("Missing IDs:", (df["Complaint ID"].str.strip() == "").sum())
    print("Missing narratives:",
          (df["Consumer complaint narrative"].str.strip() == "").sum())
    print("Earliest date:", dates.min())
    print("Latest date:", dates.max())
    print("Products:")
    print(df["Product"].value_counts().to_string())

    assert len(df) > 0
    assert dates.notna().all()
    assert df["Complaint ID"].str.strip().ne("").all()
    assert df["Consumer complaint narrative"].str.strip().ne("").all()

    if label == "January-May index pool":
        assert dates.min() >= pd.Timestamp("2026-01-01")
        assert dates.max() < pd.Timestamp("2026-06-01")
    else:
        assert dates.min() >= pd.Timestamp("2026-06-01")
        assert dates.max() < pd.Timestamp("2026-07-01")

print("\nPASS — Both CFPB datasets passed date and content validation.")



=== January-May index pool ===
Rows: 31,813
Unique complaint IDs: 31,813
Missing IDs: 0
Missing narratives: 0
Earliest date: 2026-01-01 00:00:00
Latest date: 2026-05-31 00:00:00
Products:
Product
Checking or savings account    16040
Credit card                    15773

=== June holdout ===
Rows: 6,110
Unique complaint IDs: 6,110
Missing IDs: 0
Missing narratives: 0
Earliest date: 2026-06-01 00:00:00
Latest date: 2026-06-30 00:00:00
Products:
Product
Checking or savings account    3270
Credit card                    2840

PASS — Both CFPB datasets passed date and content validation.


In [16]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/home/sagemaker-user/complaintiq")
INDEX_FILE = PROJECT / "data/processed/cfpb_index_pool_jan_may_2026.csv"

df = pd.read_csv(INDEX_FILE, dtype=str, keep_default_na=False)

duplicate_rows = df[df["Complaint ID"].duplicated(keep=False)]
duplicate_ids = duplicate_rows["Complaint ID"].nunique()

print("Total index rows:", f"{len(df):,}")
print("Unique complaint IDs:", f"{df['Complaint ID'].nunique():,}")
print("Rows with repeated IDs:", f"{len(duplicate_rows):,}")
print("Unique IDs that repeat:", f"{duplicate_ids:,}")
print("Excess duplicate rows:",
      f"{len(df) - df['Complaint ID'].nunique():,}")

if not duplicate_rows.empty:
    print("\nDuplicate occurrence counts:")
    print(
        duplicate_rows["Complaint ID"]
        .value_counts()
        .value_counts()
        .sort_index()
        .to_string()
    )

    comparison_columns = [
        c for c in df.columns if c != "Complaint ID"
    ]

    conflicts = []
    for complaint_id, group in duplicate_rows.groupby("Complaint ID"):
        if len(group[comparison_columns].drop_duplicates()) > 1:
            conflicts.append(complaint_id)

    print("\nIDs with conflicting records:", len(conflicts))

    if conflicts:
        print("Sample conflicting IDs:", conflicts[:10])
        raise RuntimeError(
            "Conflicting duplicate complaint records need investigation."
        )
else:
    print("No duplicate complaint IDs found.")


Total index rows: 31,813
Unique complaint IDs: 31,813
Rows with repeated IDs: 0
Unique IDs that repeat: 0
Excess duplicate rows: 0
No duplicate complaint IDs found.


In [17]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/home/sagemaker-user/complaintiq")
PROC = PROJECT / "data/processed"

SOURCE_FILE = PROC / "cfpb_index_pool_jan_may_2026.csv"
OUTPUT_FILE = PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv"

df = pd.read_csv(SOURCE_FILE, dtype=str, keep_default_na=False)

before = len(df)
unique_ids_before = df["Complaint ID"].nunique()

clean_df = df.drop_duplicates(
    subset=["Complaint ID"],
    keep="first"
).copy()

clean_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("Original rows:", f"{before:,}")
print("Unique IDs before cleaning:", f"{unique_ids_before:,}")
print("Rows after deduplication:", f"{len(clean_df):,}")
print("Duplicate IDs remaining:",
      clean_df["Complaint ID"].duplicated().sum())
print("Output:", OUTPUT_FILE)

assert clean_df["Complaint ID"].str.strip().ne("").all()
assert clean_df["Complaint ID"].is_unique
assert len(clean_df) == unique_ids_before

print("\nPASS — Deduplicated index pool created.")


Original rows: 31,813
Unique IDs before cleaning: 31,813
Rows after deduplication: 31,813
Duplicate IDs remaining: 0
Output: /home/sagemaker-user/complaintiq/data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv

PASS — Deduplicated index pool created.


In [18]:

import pandas as pd
from pathlib import Path

PROJECT = Path("/home/sagemaker-user/complaintiq")
PROC = PROJECT / "data/processed"

index_df = pd.read_csv(
    PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    dtype=str
)
holdout_df = pd.read_csv(
    PROC / "cfpb_holdout_june_2026.csv",
    dtype=str
)

index_ids = set(index_df["Complaint ID"].dropna())
holdout_ids = set(holdout_df["Complaint ID"].dropna())

overlap = index_ids.intersection(holdout_ids)

print("Deduplicated index records:", f"{len(index_df):,}")
print("June holdout records:", f"{len(holdout_df):,}")
print("Index unique IDs:", f"{len(index_ids):,}")
print("Holdout unique IDs:", f"{len(holdout_ids):,}")
print("Overlapping complaint IDs:", len(overlap))

assert len(overlap) == 0, "Data leakage: IDs occur in both datasets."

print("\nPASS — No complaint ID leakage between index and holdout.")


Deduplicated index records: 31,813
June holdout records: 6,110
Index unique IDs: 31,813
Holdout unique IDs: 6,110
Overlapping complaint IDs: 0

PASS — No complaint ID leakage between index and holdout.


In [19]:

from pathlib import Path
import xml.etree.ElementTree as ET
import json
import re
import html

PROJECT = Path("/home/sagemaker-user/complaintiq")
XML_DIR = PROJECT / "data/raw/ecfr"
OUTPUT = PROJECT / "data/processed/reg_sections.jsonl"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

xml_files = sorted(XML_DIR.glob("title_*_part_*.xml"))

if len(xml_files) != 6:
    raise RuntimeError(
        f"Expected 6 eCFR XML files, found {len(xml_files)}"
    )

def local_tag(element):
    return element.tag.split("}")[-1].upper()

def clean_text(value):
    value = html.unescape(value or "")
    value = re.sub(r"\s+", " ", value)
    return value.strip()

records = []
seen_ids = set()

for xml_file in xml_files:
    match = re.search(
        r"title_(\d+)_part_(\d+)_",
        xml_file.name
    )
    if not match:
        raise ValueError(f"Cannot identify title/part: {xml_file.name}")

    title_number = match.group(1)
    part_number = match.group(2)

    tree = ET.parse(xml_file)
    root = tree.getroot()

    file_count = 0

    for element in root.iter():
        if local_tag(element) != "DIV8":
            continue

        if element.attrib.get("TYPE", "").upper() != "SECTION":
            continue

        section_number = element.attrib.get("N", "").strip()
        if not section_number:
            continue

        heading_element = next(
            (child for child in element
             if local_tag(child) == "HEAD"),
            None
        )
        heading = clean_text(
            "".join(heading_element.itertext())
            if heading_element is not None else ""
        )

        body_parts = []
        for child in element:
            if local_tag(child) == "P":
                text = clean_text(" ".join(child.itertext()))
                if text:
                    body_parts.append(text)

        # Some eCFR sections contain paragraphs nested in child divisions.
        if not body_parts:
            body_parts = [
                clean_text(" ".join(p.itertext()))
                for p in element.iter()
                if local_tag(p) == "P"
            ]
            body_parts = [p for p in body_parts if p]

        body = "\n".join(body_parts)

        if not body:
            continue

        doc_id = (
            f"{title_number}CFR{part_number}_{section_number}"
            .replace("§", "")
            .replace(" ", "")
        )

        if doc_id in seen_ids:
            raise RuntimeError(f"Duplicate regulatory ID: {doc_id}")

        seen_ids.add(doc_id)

        records.append({
            "doc_id": doc_id,
            "source_type": "ecfr",
            "source_file": xml_file.name,
            "citation": (
                f"{title_number} CFR Part {part_number}, "
                f"Section {section_number}"
            ),
            "title": heading,
            "title_number": int(title_number),
            "part": part_number,
            "section": section_number,
            "page": None,
            "text": body,
            "char_count": len(body),
        })

        file_count += 1

    print(f"{xml_file.name}: {file_count} sections")

if not records:
    raise RuntimeError("No regulatory sections were extracted.")

temp_output = OUTPUT.with_suffix(".jsonl.tmp")

with temp_output.open("w", encoding="utf-8") as out:
    for record in records:
        out.write(json.dumps(record, ensure_ascii=False) + "\n")

temp_output.replace(OUTPUT)

print("\nTotal regulatory sections:", f"{len(records):,}")
print("Output:", OUTPUT)


title_12_part_1002_2026-11-03.xml: 30 sections
title_12_part_1005_2026-11-03.xml: 27 sections
title_12_part_1006_2026-11-03.xml: 15 sections
title_12_part_1022_2026-11-03.xml: 36 sections
title_12_part_1026_2026-11-03.xml: 58 sections
title_16_part_433_2026-11-03.xml: 3 sections

Total regulatory sections: 169
Output: /home/sagemaker-user/complaintiq/data/processed/reg_sections.jsonl


In [20]:

from pathlib import Path
import json

OUTPUT = Path(
    "/home/sagemaker-user/complaintiq/data/processed/reg_sections.jsonl"
)

count = 0
ids = set()
errors = []

with OUTPUT.open(encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        record = json.loads(line)
        count += 1

        doc_id = record.get("doc_id")
        if not doc_id or doc_id in ids:
            errors.append(f"Missing or duplicate doc_id at line {line_number}")
        ids.add(doc_id)

        for field in ["source_file", "citation", "text"]:
            if not record.get(field):
                errors.append(
                    f"Missing {field} at line {line_number}"
                )

print("Regulatory sections:", count)
print("Unique document IDs:", len(ids))
print("Validation errors:", len(errors))

for error in errors[:20]:
    print(error)

assert count > 0
assert not errors

print("\nPASS — eCFR extraction validated.")


Regulatory sections: 169
Unique document IDs: 169
Validation errors: 0

PASS — eCFR extraction validated.


In [21]:

from pathlib import Path
from pypdf import PdfReader
import json
import re
import logging
import tempfile

PROJECT = Path("/home/sagemaker-user/complaintiq")
PDF_DIR = PROJECT / "data/raw/pdf"
OUTPUT = PROJECT / "data/processed/pdf_pages.jsonl"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

if len(pdf_files) != 5:
    raise RuntimeError(
        f"Expected 5 PDF files, found {len(pdf_files)}"
    )

logging.getLogger("pypdf").setLevel(logging.ERROR)

def normalize_pdf_text(text):
    text = text or ""
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

total_pages = 0
extracted_pages = 0
skipped_pages = 0

temp_path = None

try:
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        newline="\n",
        dir=OUTPUT.parent,
        prefix="pdf_pages_",
        suffix=".tmp",
        delete=False,
    ) as out:
        temp_path = Path(out.name)

        for pdf_path in pdf_files:
            reader = PdfReader(str(pdf_path))
            file_extracted = 0

            for page_number, page in enumerate(reader.pages, start=1):
                total_pages += 1

                text = normalize_pdf_text(page.extract_text())

                if len(text) < 200:
                    skipped_pages += 1
                    continue

                page_id = (
                    f"{pdf_path.stem}_page_{page_number:04d}"
                )

                record = {
                    "page_id": page_id,
                    "source_type": "pdf",
                    "source_file": pdf_path.name,
                    "citation": (
                        f"{pdf_path.name}, page {page_number}"
                    ),
                    "title": pdf_path.stem,
                    "page_number": page_number,
                    "text": text,
                    "char_count": len(text),
                }

                out.write(
                    json.dumps(record, ensure_ascii=False) + "\n"
                )

                extracted_pages += 1
                file_extracted += 1

            print(
                f"{pdf_path.name}: "
                f"{len(reader.pages)} pages, "
                f"{file_extracted} retained"
            )

    temp_path.replace(OUTPUT)

except Exception:
    if temp_path and temp_path.exists():
        temp_path.unlink()
    raise

print("\n=== PDF EXTRACTION SUMMARY ===")
print("Total pages:", total_pages)
print("Retained pages:", extracted_pages)
print("Skipped pages:", skipped_pages)
print("Output:", OUTPUT)

assert total_pages > 0
assert extracted_pages > 0


exam_manual_v2_2012.pdf: 924 pages, 916 retained
sh_issue17_2018-09.pdf: 23 pages, 22 retained
sh_issue18_2019-03.pdf: 21 pages, 20 retained
sh_issue22_2020-09.pdf: 38 pages, 37 retained
sh_issue23_2021-01.pdf: 28 pages, 27 retained

=== PDF EXTRACTION SUMMARY ===
Total pages: 1034
Retained pages: 1022
Skipped pages: 12
Output: /home/sagemaker-user/complaintiq/data/processed/pdf_pages.jsonl


In [22]:

from pathlib import Path
import json

OUTPUT = Path(
    "/home/sagemaker-user/complaintiq/data/processed/pdf_pages.jsonl"
)

count = 0
page_ids = set()
errors = []

with OUTPUT.open(encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        record = json.loads(line)
        count += 1

        page_id = record.get("page_id")

        if not page_id or page_id in page_ids:
            errors.append(
                f"Missing or duplicate page ID at line {line_number}"
            )
        page_ids.add(page_id)

        if record.get("char_count", 0) < 200:
            errors.append(f"Short page at line {line_number}")

        if not record.get("source_file"):
            errors.append(f"Missing source file at line {line_number}")

        if not record.get("page_number"):
            errors.append(f"Missing page number at line {line_number}")

print("Extracted pages:", count)
print("Unique page IDs:", len(page_ids))
print("Validation errors:", len(errors))

for error in errors[:20]:
    print(error)

assert count > 0
assert not errors

print("\nPASS — PDF extraction validated.")


Extracted pages: 1022
Unique page IDs: 1022
Validation errors: 0

PASS — PDF extraction validated.


In [23]:

from pathlib import Path
import json
import re

PROJECT = Path("/home/sagemaker-user/complaintiq")
PROC = PROJECT / "data/processed"

REG_INPUT = PROC / "reg_sections.jsonl"
PDF_INPUT = PROC / "pdf_pages.jsonl"
OUTPUT = PROC / "document_chunks.jsonl"

MAX_CHARS = 3000
OVERLAP = 400

if MAX_CHARS <= OVERLAP:
    raise ValueError("MAX_CHARS must be greater than OVERLAP.")

for path in [REG_INPUT, PDF_INPUT]:
    if not path.exists():
        raise FileNotFoundError(path)

def read_jsonl(path):
    with path.open(encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {path.name}, line {line_number}"
                ) from exc

def split_text(text):
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = min(start + MAX_CHARS, text_length)

        if end < text_length:
            # Prefer ending at a sentence or word boundary.
            boundary = text.rfind(". ", start, end)
            if boundary < start + int(MAX_CHARS * 0.60):
                boundary = text.rfind(" ", start, end)

            if boundary > start:
                end = boundary + 1

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start = max(0, end - OVERLAP)

    return chunks

total_chunks = 0
source_counts = {"ecfr": 0, "pdf": 0}
chunk_ids = set()

temp_output = OUTPUT.with_suffix(".jsonl.tmp")

with temp_output.open("w", encoding="utf-8") as out:

    for input_file in [REG_INPUT, PDF_INPUT]:

        for record in read_jsonl(input_file):
            source_type = record["source_type"]

            if source_type == "ecfr":
                source_id = record["doc_id"]
            elif source_type == "pdf":
                source_id = record["page_id"]
            else:
                raise ValueError(
                    f"Unknown source type: {source_type}"
                )

            text = record.get("text", "")
            pieces = split_text(text)

            for number, piece in enumerate(pieces, start=1):
                chunk_id = f"{source_id}_chunk_{number:04d}"

                if chunk_id in chunk_ids:
                    raise RuntimeError(
                        f"Duplicate chunk ID: {chunk_id}"
                    )

                chunk_ids.add(chunk_id)

                chunk = {
                    "chunk_id": chunk_id,
                    "source_type": source_type,
                    "source_id": source_id,
                    "source_file": record["source_file"],
                    "citation": record["citation"],
                    "title": record.get("title", ""),
                    "chunk_number": number,
                    "text": piece,
                    "char_count": len(piece),
                }

                out.write(
                    json.dumps(chunk, ensure_ascii=False) + "\n"
                )

                total_chunks += 1
                source_counts[source_type] += 1

temp_output.replace(OUTPUT)

print("=== DOCUMENT CHUNKING SUMMARY ===")
print("Total chunks:", f"{total_chunks:,}")
print("eCFR chunks:", f"{source_counts['ecfr']:,}")
print("PDF chunks:", f"{source_counts['pdf']:,}")
print("Maximum chunk size:", MAX_CHARS)
print("Overlap:", OVERLAP)
print("Output:", OUTPUT)

assert total_chunks > 0


=== DOCUMENT CHUNKING SUMMARY ===
Total chunks: 1,784
eCFR chunks: 646
PDF chunks: 1,138
Maximum chunk size: 3000
Overlap: 400
Output: /home/sagemaker-user/complaintiq/data/processed/document_chunks.jsonl


In [24]:

from pathlib import Path
import json
from collections import Counter

OUTPUT = Path(
    "/home/sagemaker-user/complaintiq/data/processed/document_chunks.jsonl"
)

chunk_ids = set()
source_counts = Counter()
errors = []
max_length = 0
count = 0

with OUTPUT.open(encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            errors.append(f"Invalid JSON at line {line_number}")
            continue

        count += 1

        chunk_id = record.get("chunk_id")
        if not chunk_id or chunk_id in chunk_ids:
            errors.append(
                f"Missing or duplicate chunk ID at line {line_number}"
            )
        chunk_ids.add(chunk_id)

        for field in [
            "source_type",
            "source_id",
            "source_file",
            "citation",
            "text",
        ]:
            if not record.get(field):
                errors.append(
                    f"Missing {field} at line {line_number}"
                )

        length = len(record.get("text", ""))
        max_length = max(max_length, length)

        if length > 3000:
            errors.append(
                f"Chunk exceeds 3000 characters at line {line_number}"
            )

        if length == 0:
            errors.append(f"Empty chunk at line {line_number}")

        source_counts[record.get("source_type")] += 1

print("Total chunks:", f"{count:,}")
print("Unique chunk IDs:", f"{len(chunk_ids):,}")
print("eCFR chunks:", f"{source_counts['ecfr']:,}")
print("PDF chunks:", f"{source_counts['pdf']:,}")
print("Maximum actual chunk length:", max_length)
print("Validation errors:", len(errors))

for error in errors[:20]:
    print(error)

assert count > 0
assert count == len(chunk_ids)
assert not errors

print("\nPASS — Document chunks validated.")


Total chunks: 1,784
Unique chunk IDs: 1,784
eCFR chunks: 646
PDF chunks: 1,138
Maximum actual chunk length: 2999
Validation errors: 0

PASS — Document chunks validated.


In [25]:

from pathlib import Path
import json
import pandas as pd
import duckdb

PROJECT = Path("/home/sagemaker-user/complaintiq")
DATA = PROJECT / "data"
PROC = DATA / "processed"

required_files = [
    DATA / "complaintiq.duckdb",
    PROC / "cfpb_index_pool_jan_may_2026.csv",
    PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    PROC / "cfpb_holdout_june_2026.csv",
    PROC / "reg_sections.jsonl",
    PROC / "pdf_pages.jsonl",
    PROC / "document_chunks.jsonl",
]

print("=== REQUIRED FILES ===")
for path in required_files:
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f"Missing or empty Phase 2 output: {path}"
        )
    print(f"PASS | {path.relative_to(PROJECT)} | "
          f"{path.stat().st_size:,} bytes")

# Validate DuckDB view.
con = duckdb.connect(str(DATA / "complaintiq.duckdb"), read_only=True)

view_count = con.execute(
    "SELECT COUNT(*) FROM v_complaints"
).fetchone()[0]

print("\nDuckDB v_complaints rows:", f"{view_count:,}")
assert view_count == 37923

con.close()

# Validate complaint splits.
index_df = pd.read_csv(
    PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    dtype=str
)
holdout_df = pd.read_csv(
    PROC / "cfpb_holdout_june_2026.csv",
    dtype=str
)

overlap = (
    set(index_df["Complaint ID"].dropna())
    & set(holdout_df["Complaint ID"].dropna())
)

assert index_df["Complaint ID"].is_unique
assert len(overlap) == 0

print("Deduplicated index rows:", f"{len(index_df):,}")
print("June holdout rows:", f"{len(holdout_df):,}")
print("Overlapping complaint IDs:", len(overlap))

# Validate JSONL files and count records.
for filename in [
    "reg_sections.jsonl",
    "pdf_pages.jsonl",
    "document_chunks.jsonl",
]:
    path = PROC / filename
    count = 0

    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                json.loads(line)
                count += 1

    print(f"{filename}: {count:,} valid JSON records")
    assert count > 0

print("\n" + "=" * 70)
print("PASS — COMPLAINTIQ PHASE 2 ACCEPTANCE CHECK")
print("=" * 70)


=== REQUIRED FILES ===
PASS | data/complaintiq.duckdb | 95,432,704 bytes
PASS | data/processed/cfpb_index_pool_jan_may_2026.csv | 54,423,475 bytes
PASS | data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv | 54,423,475 bytes
PASS | data/processed/cfpb_holdout_june_2026.csv | 11,138,095 bytes
PASS | data/processed/reg_sections.jsonl | 1,473,363 bytes
PASS | data/processed/pdf_pages.jsonl | 2,624,781 bytes
PASS | data/processed/document_chunks.jsonl | 4,504,700 bytes

DuckDB v_complaints rows: 37,923
Deduplicated index rows: 31,813
June holdout rows: 6,110
Overlapping complaint IDs: 0
reg_sections.jsonl: 169 valid JSON records
pdf_pages.jsonl: 1,022 valid JSON records
document_chunks.jsonl: 1,784 valid JSON records

PASS — COMPLAINTIQ PHASE 2 ACCEPTANCE CHECK
